# AI-Text Humanizer — Starter Notebook

**Goal:** Train a model that takes AI-generated text as input and produces human-styled output.

**Dataset:** [Defactify_Text_Dataset](https://huggingface.co/datasets/Rajarshi-Roy-research/Defactify_Text_Dataset) — 73,193 samples pairing NYT human articles with AI-generated versions from Gemma-2-9B, Mistral-7B, Qwen-2-72B, LLaMA-8B, Yi-Large, GPT-4o.

**Sections:**
1. Setup & installation
2. Load dataset
3. Exploratory data analysis (EDA)
4. Build parallel (AI → Human) pairs for style-transfer training
5. Baseline linguistic feature diagnostics (perplexity/burstiness proxies)
6. Save processed splits for downstream training


## 1. Setup & Installation

In [1]:
pip install datasets --break-system-packages

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset

pd.set_option("display.max_colwidth", 120)


c:\Users\ASUS\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load Dataset

In [3]:
ds = load_dataset("Rajarshi-Roy-research/Defactify_Text_Dataset")
print(ds)                    # shows train/validation/test splits
df_train = ds["train"].to_pandas()   # convert to pandas if preferred
df_train.head()


c:\Users\ASUS\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ASUS\.cache\huggingface\hub\datasets--Rajarshi-Roy-research--Defactify_Text_Dataset. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating test split: 100%|██████████| 10963/10963 [00:00<00:00, 222

DatasetDict({
    train: Dataset({
        features: ['Text', 'Label_A', 'Label_B'],
        num_rows: 51247
    })
    validation: Dataset({
        features: ['Text', 'Label_A', 'Label_B'],
        num_rows: 10983
    })
    test: Dataset({
        features: ['Text', 'Label_A', 'Label_B'],
        num_rows: 10963
    })
})


,Text,Label_A,Label_B
0,Comments\nThe U.S. bombings thatended World War II didn’t mark theclose of atomic warfare.They were just the beginni...,0,Human_Story
1,## The Unseen Scars: The Enduring Human Toll of Nuclear Testing\n\n**By [Your Name]**\n\n**Published: [Date]**\n\nTh...,1,Gemma-2-9B
2,\n\nTitle: The Atomic Aftermath: The U.S. Bombings of World War II and the Dawn of Nuclear Warfare\n\nThe U.S. bombi...,1,Mistral-7B
3,"Human Toll of Nuclear Testing: Unraveling the Invisible LegacyIn the atomic age, the quest for nuclear prowess has l...",1,Qwen-2-72B
4,The Ghostly Whispers of Trinity The Enduring Human Toll of Nuclear Testing\n\nBy Your Name\n\nSeventyeight years ag...,1,Llama-8B


In [4]:
# Convert the other splits too, for convenience
df_val = ds["validation"].to_pandas()
df_test = ds["test"].to_pandas()

print("Train:", df_train.shape)
print("Val:  ", df_val.shape)
print("Test: ", df_test.shape)


Train: (51247, 3)
Val:   (10983, 3)
Test:  (10963, 3)


## 3. Exploratory Data Analysis

In [5]:
# Label_A: 0 = human-written, 1 = AI-generated
# Label_B: source of the text (Human_Story, Gemma-2-9B, Mistral-7B, Qwen-2-72B, etc.)

print(df_train["Label_A"].value_counts())
print()
print(df_train["Label_B"].value_counts())


Label_A
1    43926
0     7321
Name: count, dtype: int64

Label_B
Human_Story    7321
Gemma-2-9B     7321
Mistral-7B     7321
Qwen-2-72B     7321
Llama-8B       7321
Yi-Large       7321
GPT-4o         7321
Name: count, dtype: int64


In [6]:
df_train["word_count"] = df_train["Text"].str.split().apply(len)

fig, ax = plt.subplots(figsize=(8, 5))
for label, grp in df_train.groupby("Label_A"):
    ax.hist(grp["word_count"], bins=50, alpha=0.5,
            label="Human" if label == 0 else "AI-generated")
ax.set_xlabel("Word count")
ax.set_ylabel("Frequency")
ax.set_title("Text length distribution: Human vs AI-generated")
ax.legend()
plt.show()


TypeError: object of type 'float' has no len()

In [ ]:
# Sample a few rows from each class to eyeball stylistic differences
print("--- HUMAN SAMPLE ---")
print(df_train[df_train["Label_A"] == 0]["Text"].iloc[0][:600])
print()
print("--- AI SAMPLE ---")
print(df_train[df_train["Label_A"] == 1]["Text"].iloc[0][:600])


## 4. Build AI → Human Parallel Pairs

This dataset is **not natively paired at the row level** (each row is a standalone human or AI text, distinguished by `Label_A`/`Label_B`) — it was built for detection, not style transfer.

Two practical options for your humanizer training signal:

- **Option A (topic-matched pairing):** If underlying prompts/article titles are recoverable or embeddable, match AI texts to the human text they were generated from (best signal, requires more preprocessing — check `Files and versions` for any prompt/source-id columns beyond what's shown in the viewer).
- **Option B (distributional / unpaired style transfer):** Treat this as an unpaired style-transfer problem (like CycleGAN for text, or discriminator-guided decoding) using the AI set as domain X and Human set as domain Y, without row-level pairing.

The cell below sets up Option B's split, which works out of the box with this dataset as downloaded.


In [ ]:
human_texts = df_train[df_train["Label_A"] == 0]["Text"].reset_index(drop=True)
ai_texts = df_train[df_train["Label_A"] == 1]["Text"].reset_index(drop=True)

print(f"Human texts: {len(human_texts)}")
print(f"AI texts:    {len(ai_texts)}")


## 5. Baseline Linguistic Diagnostics

Detectors and humanizers both key off measurable stylistic markers. Two cheap, dependency-light proxies worth tracking as you iterate:

- **Burstiness**: variance in sentence length (AI text tends to be more uniform).
- **Type-token ratio (TTR)**: lexical diversity (AI text often has lower TTR at scale).

These give you a fast sanity check for whether your humanizer's output is actually shifting distributional style, without needing a full detector model yet.


In [ ]:
import re

def sentence_lengths(text):
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return [len(s.split()) for s in sentences if s]

def burstiness(text):
    lengths = sentence_lengths(text)
    if len(lengths) < 2:
        return np.nan
    return np.std(lengths) / (np.mean(lengths) + 1e-8)

def type_token_ratio(text):
    tokens = text.lower().split()
    if not tokens:
        return np.nan
    return len(set(tokens)) / len(tokens)

# Sample a subset for speed
sample_human = human_texts.sample(min(500, len(human_texts)), random_state=42)
sample_ai = ai_texts.sample(min(500, len(ai_texts)), random_state=42)

metrics = pd.DataFrame({
    "burstiness": pd.concat([sample_human.apply(burstiness), sample_ai.apply(burstiness)], ignore_index=True),
    "ttr": pd.concat([sample_human.apply(type_token_ratio), sample_ai.apply(type_token_ratio)], ignore_index=True),
    "source": ["Human"] * len(sample_human) + ["AI"] * len(sample_ai),
})

metrics.groupby("source")[["burstiness", "ttr"]].describe()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for source, grp in metrics.groupby("source"):
    axes[0].hist(grp["burstiness"].dropna(), bins=30, alpha=0.5, label=source)
    axes[1].hist(grp["ttr"].dropna(), bins=30, alpha=0.5, label=source)

axes[0].set_title("Burstiness (sentence-length variance)")
axes[0].legend()
axes[1].set_title("Type-Token Ratio (lexical diversity)")
axes[1].legend()
plt.tight_layout()
plt.show()


## 6. Save Processed Splits

In [ ]:
import os

os.makedirs("data/processed", exist_ok=True)

df_train.to_parquet("data/processed/train.parquet", index=False)
df_val.to_parquet("data/processed/validation.parquet", index=False)
df_test.to_parquet("data/processed/test.parquet", index=False)

human_texts.to_frame(name="text").to_csv("data/processed/human_only.csv", index=False)
ai_texts.to_frame(name="text").to_csv("data/processed/ai_only.csv", index=False)

print("Saved processed splits to data/processed/")


## Next Steps

- Decide on Option A vs. Option B pairing strategy (Section 4) based on what's actually available in the raw files.
- Pull in adversarial/paraphrase corpora (RAID, DIPPER-paraphrased sets) to supplement this dataset with texts that already model evasion-style transformations.
- Fine-tune a seq2seq model (e.g., T5/BART) or use a paraphrase-and-filter pipeline, scoring outputs against a held-out detector (e.g., a RoBERTa classifier trained on `Label_A`) to measure how far the humanized output shifts the classifier's confidence.
- Track burstiness/TTR (Section 5) plus detector confidence as your two core eval metrics across iterations.
